# NB14 — end-to-end demo

One patient, start to finish, side by side with v1 (MOFA cluster + Q4 drugs).
**Gate:** `assert_safe()` passes on every generated string.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe
try:
    import certifi, os
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception:
    pass

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = False
N_SAMPLES  = None   # full TCGA-BRCA; do not cap
N_SC_CELLS = 25_000  # Wu reference subsample if RAM is tight
N_PATIENTS = None
N_DRUGS    = None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        kwargs.setdefault("cohort", False)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
PATIENT = None  # default: first overlapping id
import json, html, numpy as np, pandas as pd
from safety import assert_safe


In [ ]:
# Load whatever artifacts exist + committed v1 tables
post = pd.read_parquet(INTERIM / "latent_posterior.parquet") if (INTERIM / "latent_posterior.parquet").exists() else None
pw = pd.read_parquet(INTERIM / "pathway_activity.parquet") if (INTERIM / "pathway_activity.parquet").exists() else None
tf = pd.read_parquet(INTERIM / "tf_activity.parquet") if (INTERIM / "tf_activity.parquet").exists() else None
syn = pd.read_parquet(INTERIM / "predicted_synergy.parquet") if (INTERIM / "predicted_synergy.parquet").exists() else None
rel = pd.read_csv(REF / "tf_reliability.parquet") if False else (pd.read_parquet(INTERIM / "tf_reliability.parquet") if (INTERIM / "tf_reliability.parquet").exists() else None)
v1_clusters = pd.read_csv(REPO_ROOT / "outputs" / "mofa" / "mofa_clusters.csv") if (REPO_ROOT / "outputs" / "mofa" / "mofa_clusters.csv").exists() else None
v1_drugs = {}
q4 = REPO_ROOT / "results" / "mofa_clusters"
if q4.exists():
    for p in q4.glob("cluster_*_drug_targets.csv"):
        v1_drugs[p.stem] = pd.read_csv(p)
nets = list((INTERIM / "causal_networks").glob("*.json"))


In [ ]:
# Choose patient
if PATIENT is None:
    if post is not None:
        PATIENT = str(post.index[0])
    elif v1_clusters is not None:
        PATIENT = str(v1_clusters.iloc[0, 0])
    else:
        PATIENT = "MB-0000"
print("patient", PATIENT)

strings = []
def emit(s):
    assert_safe(s)
    strings.append(s)
    print(s)

emit(f"Research prototype walkthrough for sample {PATIENT}. This is not clinical decision support.")
if post is not None and PATIENT in map(str, post.index):
    row = post.iloc[list(map(str, post.index)).index(PATIENT)]
    emit(f"Latent posterior width={float(row.get('width', float('nan'))):.3f}; presentation cluster={row.get('cluster', 'NA')} (posterior mass, not a clinical subtype).")
    emit("Uncertainty ellipse is the encoder posterior, not a statement of prognosis.")
if pw is not None:
    emit("Pathway activity scores are footprint inferences from expression, not measured protein activity.")
if rel is not None:
    emit("Transcription-factor estimates with low methylation reliability are flagged, not trusted as biology.")
if nets:
    emit(f"CARNIVAL produced {len(nets)} networks as an explanation overlay; they do not initialise the ODE.")
    emit("Timed-out solves are feasible but not optimal and must be flagged.")
if syn is not None and len(syn):
    top = syn.sort_values("bliss_excess", ascending=False).head(3)
    pair = f"{top.iloc[0]['drug_a']} + {top.iloc[0]['drug_b']}"
    emit(f"In-silico Bliss excess is highest for {pair} in this ODE. That is a simulation, not a trial result.")
if (ARTIFACTS / "ode_params.npz").exists():
    emit("ODE trajectories describe simulated node activity at achievable Cmax, not an observed clinical course.")
if v1_clusters is not None:
    sid = v1_clusters.columns[0]
    hit = v1_clusters[v1_clusters[sid].astype(str) == PATIENT]
    if len(hit):
        cl = int(hit.iloc[0]["MOFA_CLUSTER"])
        emit(f"v1 assigned MOFA cluster {cl} from the committed Q1 table.")
        key = f"cluster_{cl}_drug_targets"
        if key in v1_drugs and len(v1_drugs[key]):
            d0 = str(v1_drugs[key].iloc[0].get("drug", v1_drugs[key].iloc[0].iloc[1]))
            emit(f"v1 Q4 top reversing compound for that cluster is {d0} (connectivity mapping, not a treatment plan).")


In [ ]:
# GATE — every string already passed assert_safe; log it
n = len(strings)
# also verify banned ODE phrases still raise
from safety import check_safety
banned_ok = True
for phrase in ["will respond", "expected response duration", "predicted survival", "weeks of response", "time to progression"]:
    if not check_safety(phrase):
        banned_ok = False
gate("NB14", "safety_assert_safe", float(n if banned_ok else 0), 1.0,
     n=n, note=f"{n} strings checked; banned-phrase unit still active={banned_ok}")


In [ ]:
# Persist HTML
parts = ["<html><head><meta charset='utf-8'><title>v2 walkthrough</title></head><body>"]
parts.append(f"<h1>v2 single-patient walkthrough: {html.escape(str(PATIENT))}</h1>")
parts.append("<p><em>Research prototype. Not clinical decision support.</em></p><ol>")
for s in strings:
    parts.append(f"<li>{html.escape(s)}</li>")
parts.append("</ol></body></html>")
out = V2_ROOT / "reports" / "single_patient_walkthrough.html"
out.write_text("\n".join(parts))
print("wrote", out)
